In [1]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

# GRPO LLaMA 1B

ref: [willccbb/grpo_demo.py](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb)

In [11]:
# !nvidia-smi

In [5]:
import re
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

In [7]:
# load and prepare dataset

SYSTEM_PROMPT = """
Respond in the following format:

<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str:
    if "####" not in text:
        return None
    return text.split("####")[1].strip().replace(',', '').replace('$', '')

def get_gsm8k_questions(split='train') -> Dataset:
    data = load_dataset("openai/gsm8k", "main")[split]
    data = data.map(lambda x: {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            # uncomment middle messages for 1-shot prompting
            #{'role': 'user', 'content': 'What is the largest single-digit prime number?'},
            #{'role': 'assistant', 'content': XML_COT_FORMAT.format(
            #    reasoning="9 is divisble by 3 and 8 is divisible by 2, but 7 is prime.",
            #    answer="7"
            #)},
            {'role': 'user', 'content': x['question']},
        ],
        'answer': extract_hash_answer(x['answer'])
    })
    return data


dataset = get_gsm8k_questions()

ConnectionError: Couldn't reach 'openai/gsm8k' on the Hub (SSLError)

In [13]:
# reward functions

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*40, f'Question:\n{q}', f'\nAnswer:\n{answer[0]}', f'\nResponse:\n{responses[0]}', f'\nExtracted:\n{extracted_responses[0]}')
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]['content'] for completion in completions]
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses] 
    return [0.5 if match else 0.0 for match in matches]


def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001
    return count
        
def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]['content'] for completion in completions]
    return [count_xml(c) for c in contents]

In [6]:
model_id =  "Qwen/Qwen2.5-1.5B-Instruct"

if 'Llama' in model_id:
    output_dir = 'grpo_demo/outputs/Llama-1b'
    run_name = 'Llama-1b-grpo-gsm8k'
else:
    output_dir = 'grpo_demo/outputs/Qwen-1.5b'
    run_name = 'Qwen-1.5b-grpo-gsm8k'

In [7]:
training_args = GRPOConfig(
    output_dir=output_dir,
    run_name=run_name,
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    logging_steps=1,
    bf16=True,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=12,
    max_prompt_length=256,
    max_completion_length=512,
    num_train_epochs=1,
    save_steps=100,
    max_grad_norm=0.1,
    # report_to="wandb",
    log_on_each_node=False,
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    task_type="CAUSAL_LM",
    lora_dropout=0.05,
)

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2',
    device_map='auto',
    cache_dir='/hf_cache/',
).to('cuda')

tokenizer = AutoTokenizer.from_pretrained(model_id)
# 将填充标记（pad_token）设置为与序列结束标记（eos_token）相同
# 避免处理缺失的填充标记，且在推理阶段，模型通常不会生成 pad_token，因此将两者设置相同不会影响生成结果
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        xmlcount_reward_func,
        correctness_reward_func,
        int_reward_func,
        strict_format_reward_func,
        soft_format_reward_func
    ],
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config
)
trainer.train()

---------------------------------------- Question:
Ahmed and Emily are having a contest to see who can get the best grade in the class. There have been 9 assignments and Ahmed has a 91 in the class. Emily has a 92. The final assignment is worth the same amount as all the other assignments. Emily got a 90 on the final assignment. What is the minimum grade Ahmed needs to get to beat Emily if all grades are whole numbers? 
Answer:
100 
Response:
To determine the minimum grade Ahmed needs to beat Emily, we need to compare their current scores with the potential scores from the final assignment.

- Ahmed's current score: 91
- Emily's current score: 92
- Emily's score on the final assignment: 90

Total assignments: 9 assignments

Let's denote Ahmed's final score as \( A \).

- If Ahmed gets a score of 94 on the final assignment:
  - Ahmed's total: \( 91 + 94 = 185 \)
  - Emily's total: \( 92 + 90 = 182 \)
  - Ahmed beats Emily: \( 185 > 182 \)

Since both Ahmed and Emily have whole number sc

Step,Training Loss
1,0.000000
2,0.000000
3,-0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


---------------------------------------- Question:
The gauge on a water tank shows that the tank is 1/3 full of water. To fill the tank, 16 gallons of water are added. How many gallons of water does the tank hold when full? 
Answer:
24 
Response:
Let \( x \) be the total capacity of the tank in gallons. Given that the tank is \( \frac{1}{3} \) full, we have the equation:

\[ \frac{1}{3}x = \text{current volume} \]

When 16 gallons are added to fill the tank, the current volume is:

\[ \frac{1}{3}x + 16 \]

Since the tank is then full, it holds the full capacity \( x \). Thus:

\[ \frac{1}{3}x + 16 = x \]

Solving for \( x \):

\[ 16 = x - \frac{1}{3}x \]

\[ 16 = \frac{2}{3}x \]

\[ x = \frac{32}{2} \]

\[ x = 16 \times 2 \]

\[ x = 32 \]

Therefore, the tank holds 32 gallons of water when full. 
Extracted:
Let \( x \) be the total capacity of the tank in gallons. Given that the tank is \( \frac{1}{3} \) full, we have the equation:

\[ \frac{1}{3}x = \text{current volume} \]

When 16 g